# Deteksi Kualitas Buah Apel Berdasarkan Tekstur dan Warna Menggunakan Vision Transformer (ViT) dan Gradient Boosting

Notebook ini adalah modifikasi dari pipeline CNN + Gradient Boosting sebelumnya, dengan dua perubahan utama:

1. **Fokus hanya pada satu jenis buah: Apple** (fresh vs rotten). Dataset digabung dari dua sumber, lalu difilter agar hanya kelas apel yang dipakai.
2. **Ekstraktor fitur visual diganti dari CNN (VGG16) menjadi Vision Transformer (ViT)**, menggunakan model pretrained `google/vit-base-patch16-224` dari library HuggingFace `transformers`.

Gradient Boosting tetap dipertahankan sebagai classifier akhir, dan fitur tekstur (GLCM) serta warna (mean/std RGB) tetap digabungkan dengan fitur ViT, sama seperti pipeline original.

Tahapan:
1. Ekstraksi dan Preprocessing Data (filter hanya Apple)
2. Ekstraksi Fitur Tekstur (GLCM)
3. Ekstraksi Fitur Warna (Mean dan Std RGB)
4. Ekstraksi Fitur ViT (Vision Transformer pretrained)
5. Penggabungan Fitur
6. Pelatihan Gradient Boosting
7. Evaluasi Model


## 0. Setup Environment & Install Dependencies

Menginstall library `transformers` dan `torch` untuk ViT (belum termasuk default di Colab/runtime dasar).

In [ ]:
!pip install -q transformers torch torchvision --upgrade

In [ ]:
import zipfile
import os

# Sesuaikan path sesuai lokasi file zip dataset Anda di Google Drive
zip_path = "/content/drive/MyDrive/A_Jonatan/archive (1).zip"
extract_path = "/content/dataset_buah2"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)

## Fruit Quality Dataset Preprocessing (Apple Only)

Melakukan pra-pemrosesan pada Dataset Kualitas Buah, **difilter hanya untuk kelas Apple**, untuk set pelatihan dan pengujian, termasuk:
- Akuisisi & Seleksi Data (apel saja)
- Pelabelan Data (fresh = 0, rotten = 1)
- Standardisasi Ukuran Gambar
- Normalisasi Nilai Piksel
- Visualisasi Hasil


## Import Libraries yang dibutuhkan

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

print("Torch version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("✅ GPU is available:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU not available, using CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Akuisisi & Seleksi Data (Apple Only)

Muat gambar dari folder pelatihan dan pengujian, **hanya untuk kelas apel** (fresh & rotten), dari kedua sumber dataset.

> Catatan: dataset pertama (`Quality Dataset`) berstruktur `split/label/` (fresh/rotten) tanpa pemisahan per jenis buah secara eksplisit pada nama foldernya. Jika dataset ini memang berisi campuran banyak buah dalam satu folder `fresh`/`rotten`, gunakan filter nama file (mis. mengandung kata "apple") sebagai pendekatan, atau cukup andalkan dataset kedua yang sudah punya folder per buah (`freshapples`, `rottenapples`). Sesuaikan bagian `dataset1_path` di bawah jika strukturnya berbeda saat dicek langsung di Colab.

In [ ]:
# Path ke dua sumber dataset
dataset1_path = r'/content/dataset_buah2/Quality Dataset'
dataset2_path = r'/content/dataset_buah2'  # sesuaikan jika nama folder dataset kedua berbeda

def load_apple_paths_dataset1(split, label):
    # Dataset 1: struktur split/label/*.jpg. Filter nama file yang mengandung 'apple'.
    # Jika dataset ini tidak memisahkan jenis buah pada nama file, ganti logic ini
    # setelah memeriksa isi folder secara langsung (print(os.listdir(folder))).
    folder = os.path.join(dataset1_path, split, label)
    if not os.path.exists(folder):
        return []
    files = [f for f in os.listdir(folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
    apple_files = [f for f in files if 'apple' in f.lower()]
    # fallback: kalau tidak ada nama file yang mengandung 'apple', dataset ini
    # mungkin memang khusus 1 jenis buah saja -> pakai semua file
    chosen = apple_files if apple_files else files
    return [os.path.join(folder, f) for f in chosen]

def load_apple_paths_dataset2(split, label):
    # Dataset 2: struktur split/freshapples|rottenapples/*.jpg
    folder_name = 'freshapples' if label == 'fresh' else 'rottenapples'
    folder = os.path.join(dataset2_path, split, folder_name)
    if not os.path.exists(folder):
        return []
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Load paths untuk train dan test, gabungan dua dataset, hanya kelas apple
train_fresh_paths = load_apple_paths_dataset1('train', 'fresh') + load_apple_paths_dataset2('train', 'fresh')
train_rotten_paths = load_apple_paths_dataset1('train', 'rotten') + load_apple_paths_dataset2('train', 'rotten')
test_fresh_paths = load_apple_paths_dataset1('test', 'fresh') + load_apple_paths_dataset2('test', 'fresh')
test_rotten_paths = load_apple_paths_dataset1('test', 'rotten') + load_apple_paths_dataset2('test', 'rotten')

print(f"Train Fresh (Apple): {len(train_fresh_paths)} images")
print(f"Train Rotten (Apple): {len(train_rotten_paths)} images")
print(f"Test Fresh (Apple): {len(test_fresh_paths)} images")
print(f"Test Rotten (Apple): {len(test_rotten_paths)} images")

## Menggabungkan Kumpulan Data

Menggabungkan path gambar apel dari kedua dataset menjadi satu kumpulan data tunggal untuk diproses bersama.

In [ ]:
combined_train_fresh = train_fresh_paths
combined_train_rotten = train_rotten_paths
combined_test_fresh = test_fresh_paths
combined_test_rotten = test_rotten_paths

print(f"Total Train Fresh: {len(combined_train_fresh)}")
print(f"Total Train Rotten: {len(combined_train_rotten)}")
print(f"Total Test Fresh: {len(combined_test_fresh)}")
print(f"Total Test Rotten: {len(combined_test_rotten)}")

## Labeling Data

Fresh diberi label 0, sedangkan rotten (busuk) diberi label 1.

In [ ]:
label_dict = {'fresh': 0, 'rotten': 1}

all_image_paths = \
    combined_train_fresh + combined_train_rotten + \
    combined_test_fresh + combined_test_rotten

all_labels = \
    [label_dict['fresh']] * len(combined_train_fresh) + \
    [label_dict['rotten']] * len(combined_train_rotten) + \
    [label_dict['fresh']] * len(combined_test_fresh) + \
    [label_dict['rotten']] * len(combined_test_rotten)

# Shuffle paths dan labels bersamaan
all_image_paths, all_labels = shuffle(all_image_paths, all_labels, random_state=42)

print(f"Total images for processing (Apple only): {len(all_image_paths)}")
print(f"Total labels: {len(all_labels)}")
print("Sample labels (first 10):", all_labels[:10])

## 2. Standardization of Image Size

Semua ukuran gambar diseragamkan menjadi:
width: 224
height: 224
(dalam satuan pixel — ukuran ini juga sesuai dengan input size default ViT `google/vit-base-patch16-224`)

In [ ]:
def resize_image(img, size=(224, 224)):
    return cv2.resize(img, size)

resized_images = []
final_labels = []
MAX_DATA = 1255

for path, label in zip(all_image_paths, all_labels):
    if len(resized_images) >= MAX_DATA:
        break

    img = cv2.imread(path)
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized_images.append(resize_image(img))
    final_labels.append(label)

all_labels = final_labels
print(f"Jumlah data yang diproses: {len(resized_images)}")

## 3. Normalisasi Nilai Piksel

Skalakan nilai piksel ke [0, 1] dengan membaginya dengan 255. Nilai ini dipakai untuk ekstraksi fitur tekstur/warna dan visualisasi. Untuk input ke ViT, preprocessing dilakukan terpisah menggunakan `ViTImageProcessor` (lihat bagian ekstraksi fitur ViT) karena ViT punya skema normalisasi sendiri (mean/std ImageNet sesuai konfigurasi pretrained-nya).

In [ ]:
normalized_images = [img / 255.0 for img in resized_images]

print("Normalized pixel values.")

## 4. Visualisasi Hasil

Hasil gambar pada setiap langkah pemrosesan.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for i in range(3):
    axes[i, 0].imshow(resized_images[i])
    axes[i, 0].set_title('Resized (224x224)')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(normalized_images[i])
    axes[i, 1].set_title('Normalized [0,1]')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 5. Ekstraksi Fitur Tekstur dan Warna

Mengekstrak fitur tekstur menggunakan Gray-Level Co-occurrence Matrix (GLCM) dan fitur warna menggunakan mean dan standar deviasi dari channel RGB. Fitur ini nantinya digabungkan dengan fitur ViT untuk memberikan representasi yang lebih kaya untuk klasifikasi kualitas buah.

In [ ]:
def extract_texture_features(image):
    # Convert ke grayscale (image dalam skala 0-1, perlu dikonversi balik ke uint8 untuk GLCM)
    img_uint8 = (image * 255).astype(np.uint8) if image.dtype != np.uint8 else image
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    return [contrast, dissimilarity, homogeneity, energy, correlation]

def extract_color_features(image):
    means = image.mean(axis=(0, 1))
    stds = image.std(axis=(0, 1))
    return list(means) + list(stds)

texture_features = []
color_features = []

for img in normalized_images:
    texture_features.append(extract_texture_features(img))
    color_features.append(extract_color_features(img))

texture_features = np.array(texture_features)
color_features = np.array(color_features)

print(f"Texture features shape: {texture_features.shape}")
print(f"Color features shape: {color_features.shape}")

## 6. Ekstraksi Fitur menggunakan Vision Transformer (ViT)

Bagian ini menggantikan ekstraksi fitur CNN (VGG16) pada pipeline sebelumnya. Digunakan model **ViT pretrained** (`google/vit-base-patch16-224`) dari library HuggingFace `transformers` sebagai feature extractor, tanpa fine-tuning (mode evaluasi/inference saja).

Fitur diambil dari representasi token `[CLS]` pada output `last_hidden_state`, yang merupakan representasi global gambar — analog dengan output `GlobalAveragePooling2D` pada CNN sebelumnya.

In [ ]:
from transformers import ViTImageProcessor, ViTModel

vit_model_name = "google/vit-base-patch16-224"

vit_processor = ViTImageProcessor.from_pretrained(vit_model_name)
vit_model = ViTModel.from_pretrained(vit_model_name)
vit_model.to(device)
vit_model.eval()

print("ViT model loaded:", vit_model_name)
print("Hidden size (feature dimension):", vit_model.config.hidden_size)

In [ ]:
def extract_vit_features_batch(images_uint8, batch_size=16):
    # images_uint8: list of HxWx3 uint8 RGB images.
    # Mengembalikan array fitur [CLS] token, shape (N, hidden_size).
    all_features = []
    with torch.no_grad():
        for i in range(0, len(images_uint8), batch_size):
            batch = images_uint8[i:i + batch_size]
            inputs = vit_processor(images=batch, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = vit_model(**inputs)
            # CLS token representation: posisi index 0 pada last_hidden_state
            cls_features = outputs.last_hidden_state[:, 0, :]
            all_features.append(cls_features.cpu().numpy())
    return np.concatenate(all_features, axis=0)

# ViTImageProcessor menerima gambar uint8 (0-255), bukan yang sudah dinormalisasi manual
images_uint8 = [img.astype(np.uint8) for img in resized_images]

all_vit_features = extract_vit_features_batch(images_uint8, batch_size=16)
print(f"Extracted ViT features shape: {all_vit_features.shape}")

## 7. Perataan & Penskalaan Fitur

Fitur ViT sudah berbentuk vektor 1D per gambar (768 dimensi untuk `vit-base`). Standarisasikan menggunakan `StandardScaler` agar skalanya konsisten sebelum digabung dengan fitur tekstur dan warna.

In [ ]:
scaler = StandardScaler()
vit_features_scaled = scaler.fit_transform(all_vit_features)

print(f"Scaled ViT features shape: {vit_features_scaled.shape}")
print(f"Sample scaled features mean: {np.mean(vit_features_scaled, axis=0)[:5]}")

## 8. Penggabungan Fitur

Menggabungkan fitur ViT, fitur tekstur (GLCM), dan fitur warna (mean/std RGB) menjadi satu representasi fitur tunggal untuk setiap gambar.

In [ ]:
cnn_labels = np.array(all_labels[:len(resized_images)])  # nama variabel dipertahankan agar konsisten dgn pipeline GB di bawah

combined_features = np.concatenate([vit_features_scaled, texture_features, color_features], axis=1)

print(f"Combined features shape: {combined_features.shape}")

X_train_combined, X_test_combined, y_train_combined, y_test_combined = train_test_split(
    combined_features, cnn_labels, test_size=0.2, random_state=42, stratify=cnn_labels
)

print(f"Train combined features shape: {X_train_combined.shape}")
print(f"Test combined features shape: {X_test_combined.shape}")

## 9. Melatih Model Gradient Boosting

Gradient Boosting Classifier tetap dipertahankan sebagai model klasifikasi akhir, dilatih di atas fitur gabungan (ViT + tekstur + warna).

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gb_model.fit(X_train_combined, y_train_combined)

print("Gradient Boosting training completed.")

## 10. Pengujian Model

Evaluasi model ViT + Gradient Boosting pada data uji (kelas Apple: fresh vs rotten).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_gb = gb_model.predict(X_test_combined)

print("Classification Report:")
print(classification_report(y_test_combined, y_pred_gb, target_names=['fresh', 'rotten']))

print("Confusion Matrix:")
print(confusion_matrix(y_test_combined, y_pred_gb))

accuracy = gb_model.score(X_test_combined, y_test_combined)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test_combined, y_pred_gb)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['fresh', 'rotten'], yticklabels=['fresh', 'rotten'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - ViT + Gradient Boosting (Apple)')
plt.show()

## Ringkasan Perubahan dari Pipeline Original

| Aspek | Pipeline Original | Pipeline Ini |
|---|---|---|
| Jenis buah | Campuran (apple, banana, orange) | Hanya **Apple** |
| Feature extractor visual | CNN (VGG16, transfer learning) | **Vision Transformer (ViT, `google/vit-base-patch16-224`, pretrained)** |
| Representasi fitur visual | GlobalAveragePooling2D output | CLS token dari `last_hidden_state` |
| Fitur tekstur (GLCM) | Tetap | Tetap |
| Fitur warna (mean/std RGB) | Tetap | Tetap |
| Classifier akhir | Gradient Boosting | Gradient Boosting (tetap) |

Catatan tambahan:
- Karena ViT diakses lewat `transformers` (PyTorch), bagian training CNN custom (Sequential VGG16 + Dense + Dropout) pada notebook original **dihapus**, karena ViT di sini dipakai sebagai feature extractor pretrained saja (tanpa fine-tuning), sesuai pendekatan paling kompatibel dan ringan untuk dijalankan di Colab dengan GPU T4.
- Jika ingin fine-tuning ViT (bukan cuma sebagai extractor), itu memerlukan tahap training tambahan (loss, optimizer, epoch) yang bisa ditambahkan kemudian — beri tahu saya jika diperlukan.


## 11. Menyimpan Model ke Google Drive

Menyimpan tiga komponen yang diperlukan untuk inference di kemudian hari tanpa training ulang:
1. **Model Gradient Boosting** (`gb_model.pkl`) — classifier akhir.
2. **StandardScaler** (`scaler.pkl`) — dipakai untuk men-scale fitur ViT sebelum digabung dengan fitur tekstur dan warna, harus konsisten dengan saat training.
3. **Konfigurasi ViT** (`vit_config.json`) — nama model ViT pretrained, dimensi fitur, urutan fitur gabungan, dan mapping label, sehingga proses ekstraksi fitur saat inference bisa direplikasi dengan benar.

Model ViT itu sendiri tidak perlu disimpan secara manual, karena merupakan model pretrained dari HuggingFace yang bisa dipanggil ulang kapan saja menggunakan nama model yang tersimpan di `vit_config.json`.

In [ ]:
import os
import json
import joblib

# Folder tujuan di Google Drive (akan dibuat otomatis jika belum ada)
save_dir = "/content/drive/MyDrive/A_Jonatan/saved_model_vit_gb"
os.makedirs(save_dir, exist_ok=True)

# 1. Simpan model Gradient Boosting
gb_model_path = os.path.join(save_dir, "gb_model.pkl")
joblib.dump(gb_model, gb_model_path)

# 2. Simpan StandardScaler (dipakai untuk scaling fitur ViT sebelum digabung)
scaler_path = os.path.join(save_dir, "scaler.pkl")
joblib.dump(scaler, scaler_path)

# 3. Simpan info/config ViT (nama model + dimensi fitur) agar saat load tahu
#    model ViT mana yang harus dipanggil ulang dari HuggingFace
vit_config = {
    "vit_model_name": vit_model_name,
    "vit_hidden_size": vit_model.config.hidden_size,
    "image_size": 224,
    "feature_order": ["vit_cls_features", "glcm_texture_features", "color_mean_std_rgb"],
    "texture_feature_names": ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"],
    "label_mapping": {"fresh": 0, "rotten": 1},
}
vit_config_path = os.path.join(save_dir, "vit_config.json")
with open(vit_config_path, "w") as f:
    json.dump(vit_config, f, indent=2)

print("✅ Model dan konfigurasi berhasil disimpan ke Google Drive:")
print(f"  - Gradient Boosting model : {gb_model_path}")
print(f"  - StandardScaler          : {scaler_path}")
print(f"  - ViT config              : {vit_config_path}")


## 12. Memuat Ulang Model (Inference Tanpa Training Ulang)

Cell berikut menunjukkan cara memuat ulang model Gradient Boosting, StandardScaler, dan model ViT pretrained dari Google Drive/HuggingFace pada sesi Colab yang baru (misalnya setelah runtime restart), sehingga model bisa langsung dipakai untuk prediksi tanpa perlu training ulang dari awal.

Jalankan cell ini di notebook baru (setelah mount Google Drive dan install `transformers`/`torch`) untuk melakukan inference.

In [ ]:
import os
import json
import joblib
import torch
from transformers import ViTImageProcessor, ViTModel

save_dir = "/content/drive/MyDrive/A_Jonatan/saved_model_vit_gb"

# 1. Load Gradient Boosting model
gb_model_loaded = joblib.load(os.path.join(save_dir, "gb_model.pkl"))

# 2. Load StandardScaler
scaler_loaded = joblib.load(os.path.join(save_dir, "scaler.pkl"))

# 3. Load config ViT, lalu load ulang model ViT pretrained dari HuggingFace
with open(os.path.join(save_dir, "vit_config.json")) as f:
    vit_config_loaded = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

vit_processor_loaded = ViTImageProcessor.from_pretrained(vit_config_loaded["vit_model_name"])
vit_model_loaded = ViTModel.from_pretrained(vit_config_loaded["vit_model_name"])
vit_model_loaded.to(device)
vit_model_loaded.eval()

print("✅ Model dan konfigurasi berhasil dimuat ulang:")
print(f"  - Gradient Boosting model : loaded")
print(f"  - StandardScaler          : loaded")
print(f"  - ViT model               : {vit_config_loaded['vit_model_name']} (loaded ulang dari HuggingFace)")
print(f"  - Label mapping           : {vit_config_loaded['label_mapping']}")

# Contoh pakai untuk inference gambar baru:
#
# def predict_apple_quality(image_rgb_uint8):
#     # image_rgb_uint8: 1 gambar RGB uint8, ukuran berapapun (akan diresize otomatis oleh processor)
#     inputs = vit_processor_loaded(images=[image_rgb_uint8], return_tensors="pt")
#     inputs = {k: v.to(device) for k, v in inputs.items()}
#     with torch.no_grad():
#         outputs = vit_model_loaded(**inputs)
#     cls_feat = outputs.last_hidden_state[:, 0, :].cpu().numpy()
#     cls_feat_scaled = scaler_loaded.transform(cls_feat)
#
#     texture_feat = np.array([extract_texture_features(image_rgb_uint8 / 255.0)])
#     color_feat = np.array([extract_color_features(image_rgb_uint8 / 255.0)])
#
#     combined = np.concatenate([cls_feat_scaled, texture_feat, color_feat], axis=1)
#     pred = gb_model_loaded.predict(combined)[0]
#     label_map_rev = {v: k for k, v in vit_config_loaded["label_mapping"].items()}
#     return label_map_rev[pred]
